In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import optuna
import warnings
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from catboost import CatBoostClassifier
from catboost.utils import convert_to_onnx_object
from onnxruntime import InferenceSession
from tokenizers import Tokenizer
from onnx.helper import get_attribute_value
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, Int64TensorType, StringTensorType, guess_tensor_type
from skl2onnx import update_registered_converter
from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
from skl2onnx._parse import _apply_zipmap, _get_sklearn_operator_name

warnings.filterwarnings('ignore')

In [2]:
DATAPATH = "../../monitorings/meat/predictor/data/meat_dataset_v3.csv"

In [3]:
data_df = pd.read_csv(DATAPATH, index_col=0, sep=';')

data_df['product'] = data_df['product'].str.lower()
data_df['description'] = data_df['description'].str.lower()
data_df

,product,description,product_type
0,говядина,печень,печень
1,говядина,почки,почки
2,говядина,почки т/у,почки
3,говядина,путовый сустав,путовый сустав
4,говядина,рубец,рубец
...,...,...,...
67709,кура,"тушка 1 сорт,халяль ситно",тушка
67710,кура,"фарш механической обвалки,без добавок гост",фарш
67711,индейка,филе малое тм пава пава,филе грудки
67712,кура,"тушка бк с кожей,мясная",тушка


In [4]:
model_onnx = InferenceSession("../../monitorings/meat/predictor/data/models/bert/distilbert.onnx", providers=['CPUExecutionProvider'])
tokenizer = Tokenizer.from_file("../../monitorings/meat/predictor/data/models/bert/tokenizer.json")

def onnx_encode(text):
    inputs_tokens = tokenizer.encode(text)
    inputs_onnx = {
        'input_ids': np.expand_dims(np.array(inputs_tokens.ids, dtype=np.int64), axis=0),
        'attention_mask': np.expand_dims(np.array(inputs_tokens.attention_mask, dtype=np.int64), axis=0),
    }
    out_onnx = model_onnx.run(None, inputs_onnx)
    emb_onnx = np.mean(out_onnx[7], axis=1)[0]
    return emb_onnx

embeddings = [onnx_encode(text) for text in tqdm(data_df['description'].values)]
emb_df = pd.DataFrame(embeddings)

100%|██████████| 67714/67714 [10:07<00:00, 111.49it/s]


In [5]:
df_ = pd.concat(
    (data_df.reset_index(), emb_df),
    axis=1
)
df_
df_ = df_.drop(['index', 'description'], axis=1)
df_.columns = df_.columns.astype(str)

In [6]:
df_

,product,product_type,0,1,2,3,4,5,6,7,...,758,759,760,761,762,763,764,765,766,767
0,говядина,печень,-0.084378,-0.097200,0.280494,-0.291216,0.313978,0.231637,0.109654,-0.376657,...,0.014623,-0.474628,-0.205832,0.054644,-0.141018,0.086285,0.263371,0.316369,0.326258,-0.396593
1,говядина,почки,-0.073829,-0.357660,0.276923,-0.324657,0.464376,-0.156408,0.120326,-0.244890,...,-0.005219,-0.305561,-0.410204,0.194652,-0.213833,-0.258648,0.317777,0.014825,0.373704,-0.247354
2,говядина,почки,-0.006886,-0.318581,-0.099874,-0.421789,0.430274,0.175998,0.205775,-0.405187,...,0.042736,-0.646078,0.052179,-0.504751,-0.116105,-0.345333,0.149771,0.197503,0.023801,-0.551336
3,говядина,путовый сустав,-0.214793,-0.100352,0.502717,-0.303248,0.638516,0.041238,0.425129,-0.651608,...,-0.004962,-0.538822,0.166594,-0.272759,0.008362,-0.480042,-0.654679,0.101838,-0.116487,0.418954
4,говядина,рубец,0.166018,-0.125171,0.654533,-0.277937,-0.167207,0.247649,-0.003128,-0.023520,...,-0.244185,-0.576328,0.002645,-0.327170,-0.244775,-0.069312,-0.200902,-0.357261,0.370781,-0.204882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67709,кура,тушка,-0.078524,-0.024507,0.794457,-0.003212,0.348651,-0.099957,0.532231,0.056391,...,-0.055648,-0.553683,-0.154227,0.049127,0.709427,-0.490707,-0.367640,-0.101020,-0.187774,-0.221359
67710,кура,фарш,-0.107442,0.049579,0.247116,-0.056607,0.758417,0.179605,0.517171,0.156283,...,-0.544533,-0.374684,-0.850488,0.034250,-0.557336,-0.393467,-0.096283,0.431568,-0.164349,0.195104
67711,индейка,филе грудки,-0.163904,-0.016655,0.001431,0.009007,0.124267,-0.554389,0.059046,-0.104635,...,-0.216478,-0.199460,-0.402195,0.155833,0.327955,-0.154103,-0.019027,0.029506,0.030942,0.074894
67712,кура,тушка,0.059082,0.424451,0.620753,0.065953,0.152729,0.339374,0.775080,0.355443,...,-0.237648,-0.324552,-0.795738,-0.060622,0.146299,-0.210558,0.230135,0.426331,-0.091269,-0.126149


In [7]:
df_.rename(columns={"product": "category"}, inplace=True)
categories_mapping = {
    'говядина': ['Говядина', 'Телятина', 'Буйволятина', 'Кролик',  'Конина'],
    'свинина': ['Свинина', 'Свиньи'],
    'баранина': ['Баранина', 'Ягнятина'],
    'птица': ['Кура', 'Цыпленок', 'Утка', 'Индейка', 'Цесарка', 'Гусь', 'Перепел', 'Страус', 'Фазан']
}

beef_df = df_[df_['category'].str.capitalize().isin(categories_mapping['говядина'])]
pig_df = df_[df_['category'].str.capitalize().isin(categories_mapping['свинина'])]
mutton_df = df_[df_['category'].str.capitalize().isin(categories_mapping['баранина'])]
bird_df = df_[df_['category'].str.capitalize().isin(categories_mapping['птица'])]

In [8]:
# дублирование категорий с 1 экземпляром
duplicate = mutton_df[mutton_df['product_type'].isin(['бифштекс', 'вымя', 'кость', 'лопаточно-плечевая часть', 'лопаточно-шейная часть', 'лытка', 'набор для гуляша', 'набор для холодца', 'нос', 'пашина', 'пенис', 'шкура'])]
mutton_df = pd.concat((mutton_df, duplicate), axis=0)

duplicate = beef_df[beef_df['product_type'].isin(['каркас', 'крыло', 'спиной отруб', 'шпик хребтовой'])]
beef_df = pd.concat((beef_df, duplicate), axis=0)

duplicate = pig_df[pig_df['product_type'].isin(['губы', 'кожа', 'набор для чахохбили', 'подбедерок', 'спинно-поясничный отруб ', 'четвертины'])]
pig_df = pd.concat((pig_df, duplicate), axis=0)

duplicate = bird_df[bird_df['product_type'].isin(['грудинка', 'гузка', 'мясо механической обвалки', 'седло', 'филе грубки', 'шпик'])]
bird_df = pd.concat((bird_df, duplicate), axis=0)
duplicate = bird_df[bird_df['category'].isin(['фазан'])]
bird_df = pd.concat((bird_df, duplicate), axis=0)


# Тюнинг гиперпараметров

## LogReg

### beef

In [9]:
beef_le = OrdinalEncoder()
beef_df_copy = beef_df.copy()
beef_df_copy['category'] = beef_le.fit_transform(beef_df_copy['category'].values.reshape(-1, 1))
X = beef_df_copy.drop('product_type', axis=1)
y = beef_df_copy['product_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def cross_val_train_scores(model, cv):
    splits = cv.split(X_train, y_train)
    cv_scores = []

    for train_i, test_i in splits:
        X_train_fold, X_test_fold = X_train.iloc[train_i], X_train.iloc[test_i]
        y_train_fold, y_test_fold = y_train.iloc[train_i], y_train.iloc[test_i]
        model.fit(X_train_fold, y_train_fold)

        cv_scores.append(balanced_accuracy_score(y_test_fold, model.predict(X_test_fold)))
    return cv_scores

def objective(trial):
    params = {
        'class_weight':'balanced',
    }
    tune_params = {
        'C': trial.suggest_float('C', 0.0001, 2),
    }

    cv = StratifiedKFold(3, shuffle=True, random_state=42)
    scores = cross_val_train_scores(LogisticRegression(**tune_params, **params), cv=cv)

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

with open('params.txt', 'a', encoding = 'utf-8') as f:
    f.write('beef_logerg' + str(study.best_params) + str(study.best_value) + '\n')

[I 2025-07-01 18:25:51,653] A new study created in memory with name: no-name-b8812498-1e3b-49b0-8b62-fdd5b330ef81
[I 2025-07-01 18:26:14,544] Trial 0 finished with value: 0.8418841061459666 and parameters: {'C': 0.10360687432992541}. Best is trial 0 with value: 0.8418841061459666.
[I 2025-07-01 18:26:47,500] Trial 1 finished with value: 0.8360715196119343 and parameters: {'C': 1.080453163203982}. Best is trial 0 with value: 0.8418841061459666.
[I 2025-07-01 18:27:23,279] Trial 2 finished with value: 0.8365317682608064 and parameters: {'C': 1.0086658227971195}. Best is trial 0 with value: 0.8418841061459666.
[I 2025-07-01 18:27:55,714] Trial 3 finished with value: 0.834513977103057 and parameters: {'C': 1.9173858215887252}. Best is trial 0 with value: 0.8418841061459666.
[I 2025-07-01 18:28:29,377] Trial 4 finished with value: 0.8365709024370641 and parameters: {'C': 1.364821990994408}. Best is trial 0 with value: 0.8418841061459666.
[I 2025-07-01 18:29:04,507] Trial 5 finished with val

### pork

In [10]:
# свинина
pig_le = OrdinalEncoder()
pig_df_copy = pig_df.copy()
pig_df_copy['category'] = pig_le.fit_transform(pig_df_copy['category'].values.reshape(-1, 1))
X = pig_df_copy.drop('product_type', axis=1)
y = pig_df_copy['product_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def cross_val_train_scores(model, cv):
    splits = cv.split(X_train, y_train)
    cv_scores = []

    for train_i, test_i in splits:
        X_train_fold, X_test_fold = X_train.iloc[train_i], X_train.iloc[test_i]
        y_train_fold, y_test_fold = y_train.iloc[train_i], y_train.iloc[test_i]
        model.fit(X_train_fold, y_train_fold)

        cv_scores.append(balanced_accuracy_score(y_test_fold, model.predict(X_test_fold)))
    return cv_scores

def objective(trial):
    params = {
        'class_weight':'balanced',
    }
    tune_params = {
        'C': trial.suggest_float('C', 0.0001, 2),
    }

    cv = StratifiedKFold(3, shuffle=True, random_state=42)
    scores = cross_val_train_scores(LogisticRegression(**tune_params, **params), cv=cv)

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

with open('params.txt', 'a', encoding = 'utf-8') as f:
    f.write('pig_logreg' + str(study.best_params) + str(study.best_value) + '\n')


[I 2025-07-01 18:36:08,769] A new study created in memory with name: no-name-3f5003fe-bc31-4f0f-8f6c-1a71af0e9952
[I 2025-07-01 18:36:40,075] Trial 0 finished with value: 0.8449678676719391 and parameters: {'C': 1.7579692455141813}. Best is trial 0 with value: 0.8449678676719391.
[I 2025-07-01 18:37:11,856] Trial 1 finished with value: 0.8443777529504176 and parameters: {'C': 1.3826201118366734}. Best is trial 0 with value: 0.8449678676719391.
[I 2025-07-01 18:37:44,230] Trial 2 finished with value: 0.8410095790879949 and parameters: {'C': 1.87929463360325}. Best is trial 0 with value: 0.8449678676719391.
[I 2025-07-01 18:38:16,558] Trial 3 finished with value: 0.8443180549032693 and parameters: {'C': 1.1684768614853571}. Best is trial 0 with value: 0.8449678676719391.
[I 2025-07-01 18:38:51,191] Trial 4 finished with value: 0.8440898574062564 and parameters: {'C': 1.2732341760480939}. Best is trial 0 with value: 0.8449678676719391.
[I 2025-07-01 18:39:24,926] Trial 5 finished with val

### mutton

In [11]:
print("Анализ классов product_type для баранины:")
class_counts = mutton_df['product_type'].value_counts()
print(f"Всего классов: {len(class_counts)}")
print(f"\nКлассы с менее чем 5 образцами:")
print(class_counts[class_counts < 5].sort_values())
print(f"\nКлассы с 1 образцом:")
print(class_counts[class_counts == 1])

Анализ классов product_type для баранины:
Всего классов: 69

Классы с менее чем 5 образцами:
product_type
грудной отруб             1
селезенка                 1
легкое                    1
жилка                     1
мякоть                    1
уши                       1
тонкий край               1
шейно-лопаточный отруб    1
поджарка                  1
набор для студня          1
кострец                   2
толстый край              2
хвосты                    2
рагу                      2
набор для гуляша          2
бифштекс                  2
диафрагма                 2
рулька                    2
кости                     2
рубец                     2
губы                      2
набор для бульона         2
трахея                    2
медальон                  3
печень                    3
сердце                    3
спинной отруб             3
шейный отруб              3
почки                     3
калтык                    4
обрезь                    4
субпродукты               

In [12]:
# баранина
min_samples = 5
class_counts = mutton_df['product_type'].value_counts()
classes_to_duplicate = class_counts[class_counts < min_samples].index
valid_classes = class_counts[class_counts >= min_samples].index

# Подсчёт статистики
total_classes_before = len(class_counts)
classes_duplicated = len(classes_to_duplicate)
unique_classes_duplicated = len(set(classes_to_duplicate))

print(f"Всего классов до обработки: {total_classes_before}")
print(f"Классов с менее чем {min_samples} образцами: {classes_duplicated}")
print(f"Уникальных классов для дублирования: {unique_classes_duplicated}")

# Дублирование образцов для классов с недостаточным количеством
mutton_df_duplicated = mutton_df.copy()
for class_name in classes_to_duplicate:
    class_samples = mutton_df[mutton_df['product_type'] == class_name]
    needed_copies = min_samples - len(class_samples)
    
    # Дублируем образцы до достижения min_samples
    for _ in range(needed_copies):
        mutton_df_duplicated = pd.concat([mutton_df_duplicated, class_samples], ignore_index=True)

print(f"Всего образцов после дублирования: {len(mutton_df_duplicated)}")

mutton_le = OrdinalEncoder()
mutton_df_copy = mutton_df_duplicated.copy()
mutton_df_copy['category'] = mutton_le.fit_transform(mutton_df_copy['category'].values.reshape(-1, 1))
X = mutton_df_copy.drop('product_type', axis=1)
y = mutton_df_copy['product_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def cross_val_train_scores(model, cv):
    splits = cv.split(X_train, y_train)
    cv_scores = []

    for train_i, test_i in splits:
        X_train_fold, X_test_fold = X_train.iloc[train_i], X_train.iloc[test_i]
        y_train_fold, y_test_fold = y_train.iloc[train_i], y_train.iloc[test_i]
        model.fit(X_train_fold, y_train_fold)

        cv_scores.append(balanced_accuracy_score(y_test_fold, model.predict(X_test_fold)))
    return cv_scores

def objective(trial):
    params = {
        'class_weight':'balanced',
    }
    tune_params = {
        'C': trial.suggest_float('C', 0.0001, 2),
    }
    cv = StratifiedKFold(5, shuffle=True, random_state=42)
    scores = cross_val_train_scores(LogisticRegression(**tune_params, **params), cv=cv)
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

with open('params.txt', 'a', encoding = 'utf-8') as f:
    f.write('mutton_logreg' + str(study.best_params) + str(study.best_value) + '\n')

[I 2025-07-01 18:47:19,677] A new study created in memory with name: no-name-f0a651f1-a982-49e3-91d2-c65a8389426e


Всего классов до обработки: 69
Классов с менее чем 5 образцами: 33
Уникальных классов для дублирования: 33
Всего образцов после дублирования: 1005


[I 2025-07-01 18:48:12,973] Trial 0 finished with value: 0.8710045324801691 and parameters: {'C': 0.3613217052503698}. Best is trial 0 with value: 0.8710045324801691.
[I 2025-07-01 18:49:03,382] Trial 1 finished with value: 0.8675157671564607 and parameters: {'C': 0.6634411814097911}. Best is trial 0 with value: 0.8710045324801691.
[I 2025-07-01 18:50:05,140] Trial 2 finished with value: 0.8668139308663892 and parameters: {'C': 0.5627839884303237}. Best is trial 0 with value: 0.8710045324801691.
[I 2025-07-01 18:50:55,946] Trial 3 finished with value: 0.8680814555570923 and parameters: {'C': 0.2334450365641685}. Best is trial 0 with value: 0.8710045324801691.
[I 2025-07-01 18:51:29,856] Trial 4 finished with value: 0.8676879830003184 and parameters: {'C': 1.950732086337983}. Best is trial 0 with value: 0.8710045324801691.
[I 2025-07-01 18:52:20,674] Trial 5 finished with value: 0.8710045324801691 and parameters: {'C': 0.3664353041091828}. Best is trial 0 with value: 0.8710045324801691.

### bird

In [13]:
# птица
bird_le = OrdinalEncoder()
bird_df_copy = bird_df.copy()
bird_df_copy['category'] = bird_le.fit_transform(bird_df_copy['category'].values.reshape(-1, 1))
X = bird_df_copy.drop('product_type', axis=1)
y = bird_df_copy['product_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def cross_val_train_scores(model, cv):
    splits = cv.split(X_train, y_train)
    cv_scores = []

    for train_i, test_i in splits:
        X_train_fold, X_test_fold = X_train.iloc[train_i], X_train.iloc[test_i]
        y_train_fold, y_test_fold = y_train.iloc[train_i], y_train.iloc[test_i]
        model.fit(X_train_fold, y_train_fold)

        cv_scores.append(balanced_accuracy_score(y_test_fold, model.predict(X_test_fold)))
    return cv_scores

def objective(trial):
    params = {
        'class_weight':'balanced',
    }
    tune_params = {
        'C': trial.suggest_float('C', 0.0001, 2),
    }

    cv = StratifiedKFold(3, shuffle=True, random_state=42)
    scores = cross_val_train_scores(LogisticRegression(**tune_params, **params), cv=cv)
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

with open('params.txt', 'a', encoding = 'utf-8') as f:
    f.write('bird_logreg' + str(study.best_params) + str(study.best_value) + '\n')

[I 2025-07-01 19:01:02,136] A new study created in memory with name: no-name-942f6352-8ced-44da-bf92-84266809cbc5
[I 2025-07-01 19:02:04,147] Trial 0 finished with value: 0.8422424032700765 and parameters: {'C': 0.3954660267636226}. Best is trial 0 with value: 0.8422424032700765.
[I 2025-07-01 19:03:07,737] Trial 1 finished with value: 0.8321724002351815 and parameters: {'C': 1.466429165471729}. Best is trial 0 with value: 0.8422424032700765.
[I 2025-07-01 19:04:11,600] Trial 2 finished with value: 0.8330660756250268 and parameters: {'C': 1.0284422481989453}. Best is trial 0 with value: 0.8422424032700765.
[I 2025-07-01 19:05:16,566] Trial 3 finished with value: 0.8332830834437286 and parameters: {'C': 0.9979627248415986}. Best is trial 0 with value: 0.8422424032700765.
[I 2025-07-01 19:06:15,013] Trial 4 finished with value: 0.8406842934237776 and parameters: {'C': 0.151936584464412}. Best is trial 0 with value: 0.8422424032700765.
[I 2025-07-01 19:07:19,249] Trial 5 finished with val

## CatBoost

### beef

In [14]:
beef_le = OrdinalEncoder()
beef_df_copy = beef_df.copy()
beef_df_copy['category'] = beef_le.fit_transform(beef_df_copy['category'].values.reshape(-1, 1))
X = beef_df_copy.drop('product_type', axis=1)
y = beef_df_copy['product_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective(trial):
    params = {
        'verbose': False,
        'loss_function': 'MultiClassOneVsAll',
        'eval_metric': 'Accuracy',
        'task_type': 'GPU',
        'random_seed': 42,
        'border_count': 254
    }
    tune_params = {
        'depth': trial.suggest_int('depth', 3, 5),
        'iterations': trial.suggest_int('iterations', 1000, 2000),
        'learning_rate': trial.suggest_float("learning_rate", 0.03, 0.3),
        'l2_leaf_reg': trial.suggest_float("l2_leaf_reg", 1.0, 50.0),
        'random_strength': trial.suggest_float("random_strength", 1.0, 10.0),
        'fold_permutation_block': trial.suggest_int('fold_permutation_block', 2, 10),
        'auto_class_weights': trial.suggest_categorical("auto_class_weights", ["SqrtBalanced", "Balanced"]),
        # 'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
    }
    catbos_model = CatBoostClassifier(**tune_params, **params)
    catbos_model.fit(
        X=X_train,
        y=y_train,
        use_best_model=True,
        eval_set=(X_test, y_test),
        early_stopping_rounds=100
    )
    test_preds = catbos_model.predict(X_test)
    return accuracy_score(y_test, test_preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

with open('params.txt', 'a', encoding = 'utf-8') as f:
    f.write('beef_catboost' + str(study.best_params) + 'weighted accuracy' + str(study.best_value) + '\n')

[I 2025-07-01 19:20:21,938] A new study created in memory with name: no-name-480e3ada-4ac2-4e1a-b362-0ea23bde7519
[I 2025-07-01 19:23:01,081] Trial 0 finished with value: 0.9274193548387096 and parameters: {'depth': 3, 'iterations': 1386, 'learning_rate': 0.209220532717232, 'l2_leaf_reg': 29.537172564242073, 'random_strength': 9.41183317971555, 'fold_permutation_block': 8, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 0 with value: 0.9274193548387096.
[I 2025-07-01 19:25:53,374] Trial 1 finished with value: 0.9144265232974911 and parameters: {'depth': 4, 'iterations': 1030, 'learning_rate': 0.2676362130375013, 'l2_leaf_reg': 49.73538578071331, 'random_strength': 7.393408608899964, 'fold_permutation_block': 8, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 0 with value: 0.9274193548387096.
[I 2025-07-01 19:27:36,372] Trial 2 finished with value: 0.9094982078853047 and parameters: {'depth': 3, 'iterations': 1949, 'learning_rate': 0.11517061835321336, 'l2_leaf_reg': 24.571764

### pork

In [15]:
pig_le = OrdinalEncoder()
pig_df_copy = pig_df.copy()
pig_df_copy['category'] = pig_le.fit_transform(pig_df_copy['category'].values.reshape(-1, 1))
X = pig_df_copy.drop('product_type', axis=1)
y = pig_df_copy['product_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective(trial):
    params = {
        'verbose': False,
        'loss_function': 'MultiClassOneVsAll',
        'eval_metric': 'Accuracy',
        'task_type': 'GPU',
        'random_seed': 42,
        'border_count': 254
    }
    tune_params = {
        'depth': trial.suggest_int('depth', 3, 5),
        'iterations': trial.suggest_int('iterations', 1000, 2000),
        'learning_rate': trial.suggest_float("learning_rate", 0.03, 0.3),
        'l2_leaf_reg': trial.suggest_float("l2_leaf_reg", 10.0, 100.0),
        'random_strength': trial.suggest_float("random_strength", 3.0, 20.0),
        'fold_permutation_block': trial.suggest_int('fold_permutation_block', 2, 10),
        'auto_class_weights': trial.suggest_categorical("auto_class_weights", ["SqrtBalanced", "Balanced"]),
        # 'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
    }
    catbos_model = CatBoostClassifier(**tune_params, **params)
    catbos_model.fit(
        X=X_train,
        y=y_train,
        use_best_model=True,
        eval_set=(X_test, y_test),
        early_stopping_rounds=100
    )
    test_preds = catbos_model.predict(X_test)
    return accuracy_score(y_test, test_preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

with open('params.txt', 'a', encoding = 'utf-8') as f:
    f.write('pig_catboost' + str(study.best_params) + 'weighted accuracy' + str(study.best_value) + '\n')

[I 2025-07-01 20:10:40,915] A new study created in memory with name: no-name-cd41e016-82d7-4a68-bc55-5df4e2c30c36
[I 2025-07-01 20:15:37,599] Trial 0 finished with value: 0.9336714787358564 and parameters: {'depth': 5, 'iterations': 1579, 'learning_rate': 0.27351452141187227, 'l2_leaf_reg': 92.39695681777434, 'random_strength': 11.897974903583934, 'fold_permutation_block': 7, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 0 with value: 0.9336714787358564.
[I 2025-07-01 20:17:12,186] Trial 1 finished with value: 0.9289894654701522 and parameters: {'depth': 3, 'iterations': 1124, 'learning_rate': 0.23511642282660375, 'l2_leaf_reg': 53.45224820735204, 'random_strength': 11.587600586458151, 'fold_permutation_block': 2, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 0 with value: 0.9336714787358564.
[I 2025-07-01 20:18:27,805] Trial 2 finished with value: 0.887241513850956 and parameters: {'depth': 3, 'iterations': 1763, 'learning_rate': 0.049521929917296766, 'l2_leaf_reg': 68.5

### mutton

In [16]:
mutton_le = OrdinalEncoder()
mutton_df_copy = mutton_df_duplicated.copy()
mutton_df_copy['category'] = mutton_le.fit_transform(mutton_df_copy['category'].values.reshape(-1, 1))
X = mutton_df_copy.drop('product_type', axis=1)
y = mutton_df_copy['product_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective(trial):
    params = {
        'verbose': False,
        'loss_function': 'MultiClassOneVsAll',
        'eval_metric': 'Accuracy',
        'task_type': 'GPU',
        'random_seed': 42,
        'border_count': 254
    }
    tune_params = {
        'depth': trial.suggest_int('depth', 3, 5),
        'iterations': trial.suggest_int('iterations', 100, 2000),
        'learning_rate': trial.suggest_float("learning_rate", 0.03, 0.3),
        'l2_leaf_reg': trial.suggest_float("l2_leaf_reg", 10.0, 100.0),
        'random_strength': trial.suggest_float("random_strength", 5.0, 30.0),
        'fold_permutation_block': trial.suggest_int('fold_permutation_block', 2, 10),
        'auto_class_weights': trial.suggest_categorical("auto_class_weights", ["SqrtBalanced", "Balanced"]),
        # 'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
    }
    catbos_model = CatBoostClassifier(**tune_params, **params)
    catbos_model.fit(
        X=X_train,
        y=y_train,
        use_best_model=True,
        eval_set=(X_test, y_test),
        early_stopping_rounds=100
    )
    test_preds = catbos_model.predict(X_test)
    return accuracy_score(y_test, test_preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

with open('params.txt', 'a', encoding = 'utf-8') as f:
    f.write('mutton_catboost' + str(study.best_params) + 'weighted accuracy' + str(study.best_value) + '\n')

[I 2025-07-01 20:53:52,716] A new study created in memory with name: no-name-ba90a7a2-4767-459e-87c7-c7dc6de29528
[I 2025-07-01 20:54:33,503] Trial 0 finished with value: 0.8507462686567164 and parameters: {'depth': 4, 'iterations': 1346, 'learning_rate': 0.2711634802643364, 'l2_leaf_reg': 21.81611212904169, 'random_strength': 25.751880641395275, 'fold_permutation_block': 8, 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.8507462686567164.
[I 2025-07-01 20:55:23,706] Trial 1 finished with value: 0.8805970149253731 and parameters: {'depth': 5, 'iterations': 1773, 'learning_rate': 0.21331734803807034, 'l2_leaf_reg': 10.163533732195859, 'random_strength': 7.893342654392297, 'fold_permutation_block': 5, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 1 with value: 0.8805970149253731.
[I 2025-07-01 20:56:14,471] Trial 2 finished with value: 0.835820895522388 and parameters: {'depth': 5, 'iterations': 846, 'learning_rate': 0.22925181399524358, 'l2_leaf_reg': 78.45986551

### bird

In [17]:
bird_le = OrdinalEncoder()
bird_df_copy = bird_df.copy()
bird_df_copy['category'] = bird_le.fit_transform(bird_df_copy['category'].values.reshape(-1, 1))
X = bird_df_copy.drop('product_type', axis=1)
y = bird_df_copy['product_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective(trial):
    params = {
        'verbose': False,
        'loss_function': 'MultiClassOneVsAll',
        'eval_metric': 'Accuracy',
        'task_type': 'GPU',
        'random_seed': 42,
        'border_count': 254
    }
    tune_params = {
        'depth': trial.suggest_int('depth', 3, 5),
        'iterations': trial.suggest_int('iterations', 1000, 2000),
        'learning_rate': trial.suggest_float("learning_rate", 0.03, 0.3),
        'l2_leaf_reg': trial.suggest_float("l2_leaf_reg", 1.0, 50.0),
        'random_strength': trial.suggest_float("random_strength", 1.0, 10.0),
        'fold_permutation_block': trial.suggest_int('fold_permutation_block', 2, 10),
        'auto_class_weights': trial.suggest_categorical("auto_class_weights", ["SqrtBalanced", "Balanced"]),
        # 'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
    }
    catbos_model = CatBoostClassifier(**tune_params, **params)
    catbos_model.fit(
        X=X_train,
        y=y_train,
        use_best_model=True,
        eval_set=(X_test, y_test),
        early_stopping_rounds=100
    )
    test_preds = catbos_model.predict(X_test)
    return accuracy_score(y_test, test_preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

with open('params.txt', 'a', encoding = 'utf-8') as f:
    f.write('bird_catboost_' + str(study.best_params) + 'weighted accuracy' + str(study.best_value) + '\n')

[I 2025-07-01 21:25:23,722] A new study created in memory with name: no-name-58523735-732d-4271-bad0-bf4b34972864
[I 2025-07-01 21:26:14,262] Trial 0 finished with value: 0.9235137327616177 and parameters: {'depth': 5, 'iterations': 1351, 'learning_rate': 0.2553043047830163, 'l2_leaf_reg': 15.432087539855766, 'random_strength': 4.898379724358179, 'fold_permutation_block': 5, 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.9235137327616177.
[I 2025-07-01 21:27:02,271] Trial 1 finished with value: 0.9071734847606907 and parameters: {'depth': 5, 'iterations': 1528, 'learning_rate': 0.16293065550822405, 'l2_leaf_reg': 11.283587805570882, 'random_strength': 4.877663604917267, 'fold_permutation_block': 9, 'auto_class_weights': 'Balanced'}. Best is trial 0 with value: 0.9235137327616177.
[I 2025-07-01 21:30:12,823] Trial 2 finished with value: 0.9662765094448951 and parameters: {'depth': 4, 'iterations': 1557, 'learning_rate': 0.23735692203721878, 'l2_leaf_reg': 38.5480017132

# Сохранение моделей

In [ ]:
for df, reg_power, name in [(beef_df, 0.059269555634620164, 'Говядина'), (pig_df, 0.25481644439042445, 'Свинина'), (mutton_df, 0.3613217052503698, 'Баранина'), (bird_df, 0.03602531078286672, 'Птица')]:
    # old: {beef_df: 0.42335358835241554, pig_df: 0.46292901978098716, mutton_df: 0.13090857438995906, bird_df: 0.0973232033878918
    
    # из-за сильного классового дисбаланса дублирую сэмплы баранины
    df_balanced = df.copy()
    if name == 'Баранина':
        problem_classes = df_balanced[df_balanced['category'].isin(['баранина', 'ягнятина'])]
        for _ in range(3):
            df_balanced = pd.concat([df_balanced, problem_classes], ignore_index=True)
    
    preprocessor = ColumnTransformer([('text', OrdinalEncoder(), [0])], remainder='passthrough')
    pipe = Pipeline(
        steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression(class_weight='balanced', C=reg_power))]
    )
    X = df_balanced.drop('product_type', axis=1)
    y = df_balanced['product_type']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12, stratify=y)

    pipe.fit(X, y)

    test_preds = pipe.predict(X_test)
    print('test balanced_accuracy', balanced_accuracy_score(y_test, test_preds), 'test accuracy', accuracy_score(y_test, test_preds))

    initial_type = [('str_input', StringTensorType([None, 1])),
                    ('float_input', FloatTensorType([None, 768]))]
    model_onnx = convert_sklearn(pipe, initial_types=initial_type)
    with open(f"{name}_logreg.onnx", "wb") as f:
        f.write(model_onnx.SerializeToString())

test balanced_accuracy 0.9804031581481781 test accuracy 0.953405017921147
test balanced_accuracy 0.9946961464528634 test accuracy 0.9840031213421772
test balanced_accuracy 0.999809305873379 test accuracy 0.9985029940119761
test balanced_accuracy 0.9833035743570144 test accuracy 0.974968130721984


### catboost

Для конвертации в onnx необходимо написать свои функции конвертации для catboost. Код взят с официального туториала https://onnx.ai/sklearn-onnx/auto_tutorial/plot_gexternal_catboost.html

In [29]:
def skl2onnx_parser_castboost_classifier(scope, model, inputs, custom_parsers=None):
    options = scope.get_options(model, dict(zipmap=True))
    no_zipmap = isinstance(options["zipmap"], bool) and not options["zipmap"]

    alias = _get_sklearn_operator_name(type(model))
    this_operator = scope.declare_local_operator(alias, model)
    this_operator.inputs = inputs

    label_variable = scope.declare_local_variable("label", Int64TensorType())
    prob_dtype = guess_tensor_type(inputs[0].type)
    probability_tensor_variable = scope.declare_local_variable(
        "probabilities", prob_dtype
    )
    this_operator.outputs.append(label_variable)
    this_operator.outputs.append(probability_tensor_variable)
    probability_tensor = this_operator.outputs

    if no_zipmap:
        return probability_tensor

    return _apply_zipmap(
        options["zipmap"], scope, model, inputs[0].type, probability_tensor
    )


def skl2onnx_convert_catboost(scope, operator, container):
    """
    CatBoost returns an ONNX graph with a single node.
    This function adds it to the main graph.
    """
    onx = convert_to_onnx_object(operator.raw_operator)
    opsets = {d.domain: d.version for d in onx.opset_import}
    if "" in opsets and opsets[""] >= container.target_opset:
        raise RuntimeError("CatBoost uses an opset more recent than the target one.")
    if len(onx.graph.initializer) > 0 or len(onx.graph.sparse_initializer) > 0:
        raise NotImplementedError(
            "CatBoost returns a model initializers. This option is not implemented yet."
        )
    if (
        len(onx.graph.node) not in (1, 2)
        or not onx.graph.node[0].op_type.startswith("TreeEnsemble")
        or (len(onx.graph.node) == 2 and onx.graph.node[1].op_type != "ZipMap")
    ):
        types = ", ".join(map(lambda n: n.op_type, onx.graph.node))
        raise NotImplementedError(
            f"CatBoost returns {len(onx.graph.node)} != 1 (types={types}). "
            f"This option is not implemented yet."
        )
    node = onx.graph.node[0]
    atts = {}
    for att in node.attribute:
        atts[att.name] = get_attribute_value(att)
    container.add_node(
        node.op_type,
        [operator.inputs[0].full_name],
        [operator.outputs[0].full_name, operator.outputs[1].full_name],
        op_domain=node.domain,
        op_version=opsets.get(node.domain, None),
        **atts,
    )


update_registered_converter(
    CatBoostClassifier,
    "CatBoostCatBoostClassifier",
    calculate_linear_classifier_output_shapes,
    skl2onnx_convert_catboost,
    parser=skl2onnx_parser_castboost_classifier,
    options={"nocl": [True, False], "zipmap": [True, False, "columns"]},
)

In [31]:
train_data = [
    (
        beef_df,
        {
            'depth': 3, 'iterations': 1961, 'learning_rate': 0.24211334049159006, 'l2_leaf_reg': 7.0422324402524605, 
            'random_strength': 9.325586023176506, 'fold_permutation_block': 4, 'auto_class_weights': 'SqrtBalanced'
        },
        'Говядина'
    ),
    (
        pig_df,
        {
            'depth': 5, 'iterations': 1664, 'learning_rate': 0.21384214801999102, 'l2_leaf_reg': 31.460470832453527, 
            'random_strength': 17.770338490574982, 'fold_permutation_block': 4, 'auto_class_weights': 'SqrtBalanced'
        },
        'Свинина'
    ),
    (
        mutton_df,
        {
            'depth': 5, 'iterations': 1299, 'learning_rate': 0.25338047694849597, 'l2_leaf_reg': 10.39306565177209, 
            'random_strength': 26.057807222448986, 'fold_permutation_block': 7, 'auto_class_weights': 'SqrtBalanced'
        },
        'Баранина'
    ),
    (
        bird_df,
        {
            'depth': 4, 'iterations': 1557, 'learning_rate': 0.23735692203721878, 'l2_leaf_reg': 38.54800171324363, 
            'random_strength': 3.520632998532285, 'fold_permutation_block': 2, 'auto_class_weights': 'SqrtBalanced'
        },
        'Птица'
    ),
]


for df, params, name in train_data:

    # из-за сильного классового дисбаланса дублирую сэмплы баранины
    df_balanced = df.copy()
    if name == 'Баранина':
        problem_classes = df_balanced[df_balanced['category'].isin(['баранина', 'ягнятина'])]
        for _ in range(3):
            df_balanced = pd.concat([df_balanced, problem_classes], ignore_index=True)

    X = df_balanced.drop('product_type', axis=1)
    y = df_balanced['product_type']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


    const_params = {
        'verbose': False,
        'loss_function': 'MultiClassOneVsAll',
        'eval_metric': 'Accuracy',
        'task_type': 'GPU',
        'random_seed': 42,
        'border_count': 254,
    }

    preprocessor = ColumnTransformer([('text', OrdinalEncoder(), [0])], remainder='passthrough')
    preprocessor.fit(X)
    pipe = Pipeline(
        steps=[("preprocessor", preprocessor), ("classifier", CatBoostClassifier(**const_params, **params))]
    )

    pipe.fit(
        X=X_train,
        y=y_train,
        classifier__use_best_model=True,
        classifier__eval_set=(preprocessor.transform(X_test), y_test),
        classifier__early_stopping_rounds=100
    )

    test_preds = pipe.predict(X_test)
    print('test balanced_accuracy', balanced_accuracy_score(y_test, test_preds), 'test accuracy', accuracy_score(y_test, test_preds))


    initial_type = [('str_input', StringTensorType([None, 1])),
                ('float_input', FloatTensorType([None, 768]))]

    model_onnx = convert_sklearn(
        pipe,
        f"pipeline_catboost",
        initial_types=initial_type,
    )
    with open(f"{name}_catboost.onnx", "wb") as f:
        f.write(model_onnx.SerializeToString())

test balanced_accuracy 0.8017892579731764 test accuracy 0.9278673835125448
test balanced_accuracy 0.768240413852875 test accuracy 0.9371829886851346
test balanced_accuracy 0.9975845410628018 test accuracy 0.9985029940119761
test balanced_accuracy 0.8209066702051491 test accuracy 0.9662765094448951
